In [104]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder , MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier ,StackingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix , precision_score , recall_score,f1_score

In [68]:
data = pd.read_csv("Loan_Approval_Dataset.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3192 entries, 0 to 3191
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Loan_ID             3192 non-null   object 
 1   Gender              3115 non-null   object 
 2   Married             3126 non-null   object 
 3   Dependents          3192 non-null   int64  
 4   Education           3126 non-null   object 
 5   Employment_Status   3136 non-null   object 
 6   Applicant_Income    3157 non-null   float64
 7   Coapplicant_Income  3192 non-null   float64
 8   Loan_Amount         3151 non-null   float64
 9   Loan_Term           3192 non-null   int64  
 10  Credit_History      3192 non-null   int64  
 11  Property_Area       3192 non-null   object 
 12  Age                 3192 non-null   float64
 13  Loan_Status         3192 non-null   object 
dtypes: float64(4), int64(3), object(7)
memory usage: 349.3+ KB


In [69]:
df = data.copy()

In [70]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Employment_Status,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Term,Credit_History,Property_Area,Age,Loan_Status
0,LN03437,Female,Yes,3,Graduate,Salaried,34473.911669,13914.721574,240732.458366,180,1,Semiurban,40.084028,Approved
1,LN04823,Female,Yes,1,Graduate,Self-Employed,59718.020682,18720.814787,122547.278442,180,1,Semiurban,47.243644,Approved
2,LN04186,Female,Yes,1,Not Graduate,Salaried,106981.515734,2229.154377,140550.937774,60,1,Rural,24.292070,Approved
3,LN00380,Male,Yes,2,Graduate,Salaried,43205.563156,35578.111771,146135.738127,120,1,Semiurban,38.742244,Approved
4,LN04698,Female,Yes,0,Graduate,Salaried,62526.283990,20212.157912,203408.051744,180,1,Semiurban,41.164721,Approved


In [71]:
df.drop("Loan_ID",axis=1,inplace=True)
df.isnull().sum()

Gender                77
Married               66
Dependents             0
Education             66
Employment_Status     56
Applicant_Income      35
Coapplicant_Income     0
Loan_Amount           41
Loan_Term              0
Credit_History         0
Property_Area          0
Age                    0
Loan_Status            0
dtype: int64

In [72]:
df["Gender"].fillna(df["Gender"].mode()[0], inplace=True)
df["Married"].fillna(df["Married"].mode()[0], inplace=True)
df["Education"].fillna(df["Education"].mode()[0], inplace=True)
df["Employment_Status"].fillna(df["Employment_Status"].mode()[0], inplace=True)
df["Applicant_Income"].fillna(df["Applicant_Income"].mean(), inplace=True)
df["Loan_Amount"].fillna(df["Loan_Amount"].mean(), inplace=True)

C:\Users\shado\AppData\Local\Temp\ipykernel_14400\1222511586.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Gender"].fillna(df["Gender"].mode()[0], inplace=True)
C:\Users\shado\AppData\Local\Temp\ipykernel_14400\1222511586.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

In [73]:
df.isnull().sum()

Gender                0
Married               0
Dependents            0
Education             0
Employment_Status     0
Applicant_Income      0
Coapplicant_Income    0
Loan_Amount           0
Loan_Term             0
Credit_History        0
Property_Area         0
Age                   0
Loan_Status           0
dtype: int64

In [74]:
print(df["Education"].unique())
print(df["Employment_Status"].unique())
print(df["Property_Area"].unique())
print(df["Loan_Status"].unique())

['Graduate' 'Not Graduate']
['Salaried' 'Self-Employed' 'Unemployed']
['Semiurban' 'Rural' 'Urban']
['Approved' 'Rejected']


In [75]:
labelencoder = LabelEncoder()
columns = ["Education","Gender","Married","Loan_Status"]
for col in columns:
    df[col] = labelencoder.fit_transform(df[col])

In [76]:
df = pd.get_dummies(df,columns=["Property_Area"],dtype=int)
df = pd.get_dummies(df,columns=["Employment_Status"],dtype=int)

In [77]:
df["Age"] = df["Age"].astype("int64")
df.head()

,Gender,Married,Dependents,Education,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Term,Credit_History,Age,Loan_Status,Property_Area_Rural,Property_Area_Semiurban,Property_Area_Urban,Employment_Status_Salaried,Employment_Status_Self-Employed,Employment_Status_Unemployed
0,0,1,3,0,34473.911669,13914.721574,240732.458366,180,1,40,0,0,1,0,1,0,0
1,0,1,1,0,59718.020682,18720.814787,122547.278442,180,1,47,0,0,1,0,0,1,0
2,0,1,1,1,106981.515734,2229.154377,140550.937774,60,1,24,0,1,0,0,1,0,0
3,1,1,2,0,43205.563156,35578.111771,146135.738127,120,1,38,0,0,1,0,1,0,0
4,0,1,0,0,62526.283990,20212.157912,203408.051744,180,1,41,0,0,1,0,1,0,0


In [78]:
y = df["Loan_Status"]
X = df.drop("Loan_Status",axis=1)

In [81]:
minmax = MinMaxScaler()
minmaxscaled = minmax.fit_transform(X)
df = pd.DataFrame(minmaxscaled,columns=X.columns)

In [84]:
df_train ,df_test , y_train , y_test = train_test_split(df , y , test_size=0.2 , random_state=42)

In [88]:
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [ ]:
estimators = [
    ("rf", RandomForestClassifier(n_estimators=100,random_state=42)),
    ("xgb",XGBClassifier(random_state=42)),
    ("svc",SVC(C=1,probability=True,random_state=42))
]

In [96]:
stacking_model = StackingClassifier(estimators=estimators,final_estimator=LogisticRegression(random_state=42),cv=5)

In [97]:
stacking_model.fit(df_train,y_train)

StackingClassifier(cv=5,
                   estimators=[('rf', RandomForestClassifier(random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              gro...
                                              learning_rate=None, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None,
                                              max_depth=None, max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=None, n_jobs=None,
                                              num_parallel_tree=None, ...)),
                               ('svc', SVC(probability=True, random_state=42))],
                   final_estimator=LogisticRegression(random_state=42))

In [107]:
predicted_result = stacking_model.predict(df_test)
accuracy = accuracy_score(y_test,predicted_result)
precision = precision_score(y_test,predicted_result)
recall = recall_score(y_test,predicted_result)
f1score = f1_score(y_test,predicted_result)
Confusion_matrix = confusion_matrix(y_test,predicted_result)
print(f"Accuracy is {accuracy}")
print(f"Precision is {precision}")
print(f"Recall is {recall}")
print(f"F1_score is {f1score}")
print(f"confusion matrix is \n{Confusion_matrix}")

Accuracy is 0.9843505477308294
Precision is 0.9878048780487805
Recall is 0.9818181818181818
F1_score is 0.9848024316109423
confusion matrix is 
[[305   4]
 [  6 324]]
